# Books Web Scraping and Analysis

Beginner project using Books to Scrape.

## Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
from urllib.parse import urljoin

## Scrape Books from All Pages

In [ ]:
books = []

for page in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    book_items = soup.find_all("article", class_="product_pod")

    for book in book_items:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating = book.find("p", class_="star-rating")["class"][1]
        availability = book.find("p", class_="instock availability").text.strip()
        product_url = urljoin(url, book.h3.a["href"])

        books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Product_URL": product_url
        })

print("Total books scraped:", len(books))

## Create DataFrame

In [ ]:
df = pd.DataFrame(books)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Check Missing Values and Duplicates

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## Clean Price and Rating

In [ ]:
df["Price"] = df["Price"].str.replace("£", "", regex=False).astype(float)

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["Rating"] = df["Rating"].map(rating_map)
df.head()

## Save Data to CSV

In [ ]:
df.to_csv("books_data.csv", index=False)
print("books_data.csv saved successfully")

## Basic Analysis

In [ ]:
print("Average price:", round(df["Price"].mean(), 2))
print("Highest price:", df["Price"].max())
print("Lowest price:", df["Price"].min())

In [ ]:
df["Rating"].value_counts().sort_index()

## Top 10 Most Expensive Books

In [ ]:
df.sort_values("Price", ascending=False).head(10)[["Title", "Price", "Rating"]]

## Price Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["Price"], bins=10, edgecolor="black")
plt.xlabel("Price")
plt.ylabel("Number of Books")
plt.title("Book Price Distribution")
plt.show()

## Books by Rating

In [ ]:
rating_counts = df["Rating"].value_counts().sort_index()

plt.figure(figsize=(7, 5))
plt.bar(rating_counts.index, rating_counts.values)
plt.xlabel("Rating")
plt.ylabel("Number of Books")
plt.title("Number of Books by Rating")
plt.xticks([1, 2, 3, 4, 5])
plt.show()